
# QCVV Residual Analysis: Beyond Exponential Decay

This notebook extends **Notebook 64** by making the **RB residuals** the primary object of study.

## Goals
- Simulate structured fidelity decay for a neutral-atom CZ gate
- Fit a standard RB curve: $A p^m + B$
- Study residual structure directly
- Smooth residuals to reveal non-random behavior
- Use a simple spectral view (FFT magnitude) to expose structured components
- Track how residual amplitude changes with the effective noise coordinate
  $\gamma_{\mathrm{eff}} = \gamma + \lambda \gamma_{\phi}$

The central idea is simple:

> if RB residuals are structured rather than random, then a single exponential
> decay parameter does not capture the whole noise story.



## Imports


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

np.random.seed(11)


In [ ]:
import sys
from pathlib import Path

# Adjust this path as needed depending on notebook location
sys.path.append(str(Path("../../triplet-phase-lock/src").resolve()))

from tpl.classify import classify_consistency, classify_with_reason
from tpl.stages import stage_name



## Model setup

We again define the effective coordinate
\[
\gamma_{\mathrm{eff}} = \gamma + \lambda \gamma_{\phi},
\]
and construct a synthetic fidelity signal with:
- a dominant exponential envelope,
- a small coherent oscillatory correction,
- a weak drift term.

This is a compact way to mimic structured open-system behavior that can produce
systematic deviations from simple RB fits.


## Triplet-Phase-Lock interpretation

This notebook studies residual structure in a synthetic Rydberg QCVV setting.

Triplet-phase-lock gives a simple interpretation layer:

- **VC** → residual behavior remains consistent with constrained structure
- **IA** → residual behavior indicates mismatch, instability, or structured deviation beyond a simple fit

Here we use the framework as a lightweight classifier on fit-quality and residual metrics, not as a replacement for the physical model.


In [ ]:

gamma = 0.018
gamma_phi = 0.010
lam = 0.75
gamma_eff = gamma + lam * gamma_phi

Omega = 1.00
Delta = 0.35
V = 1.20

m = np.arange(0, 61)

A_true = 0.48
B_true = 0.50
coherent_amp = 0.020
coherent_freq = 0.22
drift_amp = 0.00006
noise_sigma = 0.003

gamma_eff


In [ ]:

fidelity_true = (
    A_true * np.exp(-gamma_eff * m) + B_true
    + coherent_amp * np.cos(coherent_freq * m) * np.exp(-0.015 * m)
    - drift_amp * (m**2)
)

fidelity_obs = fidelity_true + np.random.normal(0.0, noise_sigma, size=len(m))
fidelity_obs = np.clip(fidelity_obs, 0.0, 1.0)



## Synthetic decay


In [ ]:

plt.figure(figsize=(8, 4.8))
plt.plot(m, fidelity_true, label="Synthetic underlying decay")
plt.scatter(m, fidelity_obs, s=18, label="Observed data")
plt.xlabel("Gate depth $m$")
plt.ylabel("Fidelity")
plt.title("Rydberg CZ gate: structured synthetic decay")
plt.legend()
plt.tight_layout()
plt.show()



## Standard RB fit

We fit
\[
F_{\mathrm{RB}}(m) = A p^m + B.
\]

This compresses the behavior into a single decay parameter $p$.
If residuals retain structure after the fit, then the decay is not fully described
by a simple exponential RB model.


In [ ]:

def rb_model(m, A, p, B):
    return A * (p ** m) + B

p0 = (0.45, 0.97, 0.50)
bounds = ([0.0, 0.0, 0.0], [1.0, 1.0, 1.0])

params_rb, cov_rb = curve_fit(rb_model, m, fidelity_obs, p0=p0, bounds=bounds, maxfev=20000)
A_fit, p_fit, B_fit = params_rb
fidelity_rb = rb_model(m, A_fit, p_fit, B_fit)

A_fit, p_fit, B_fit


In [ ]:

plt.figure(figsize=(8, 4.8))
plt.scatter(m, fidelity_obs, s=18, label="Observed data")
plt.plot(m, fidelity_rb, linewidth=2.2, label=f"RB fit: A p^m + B,  p={p_fit:.4f}")
plt.xlabel("Gate depth $m$")
plt.ylabel("Fidelity")
plt.title("Standard RB-style fit")
plt.legend()
plt.tight_layout()
plt.show()



## Residual analysis

The residuals are:
\[
r(m) = F_{\mathrm{obs}}(m) - F_{\mathrm{RB}}(m).
\]

If these residuals were just random scatter around zero, the RB model would be
capturing the signal well. If they show curvature, oscillation, or drift, that is
evidence for structured noise beyond a single-parameter decay picture.


In [ ]:

residuals = fidelity_obs - fidelity_rb
rmse_rb = np.sqrt(np.mean(residuals**2))

plt.figure(figsize=(8, 4.4))
plt.axhline(0.0, linewidth=1.2)
plt.scatter(m, residuals, s=18)
plt.xlabel("Gate depth $m$")
plt.ylabel("Residual")
plt.title(f"RB residuals (RMSE = {rmse_rb:.5f})")
plt.tight_layout()
plt.show()

rmse_rb


In [ ]:
# Triplet-phase-lock classification of the first fit
# Lower residual RMSE is treated as more structurally consistent.
# We convert RMSE to a simple consistency score in [0, 1].

consistency_score = max(0.0, 1.0 - rmse_rb / 0.02)
tpl_label = classify_consistency(consistency_score, threshold=0.9)
tpl_summary = classify_with_reason(consistency_score, threshold=0.9)

print("Triplet stage:", stage_name(1))
print("Consistency score:", round(consistency_score, 4))
print("TPL label:", tpl_label)
print("TPL summary:", tpl_summary)



## Smoothed residuals

A short moving average helps reveal whether the residuals have coherent structure
rather than behaving like independent random noise.


In [ ]:

window = 5
kernel = np.ones(window) / window
residuals_smooth = np.convolve(residuals, kernel, mode="same")

plt.figure(figsize=(8, 4.5))
plt.axhline(0.0, linewidth=1.2)
plt.plot(m, residuals, marker="o", linestyle="-", label="Residuals")
plt.plot(m, residuals_smooth, linewidth=2.4, label=f"{window}-point moving average")
plt.xlabel("Gate depth $m$")
plt.ylabel("Residual")
plt.title("Residual structure beyond simple RB decay")
plt.legend()
plt.tight_layout()
plt.show()



## Interpretation of RB residuals

The structured residual pattern indicates that the decay is not purely exponential.

In randomized benchmarking, ideal Markovian noise produces approximately exponential decay.
Systematic deviations from this form suggest:
- coherent error contributions,
- non-Markovian effects,
- parameter drift or control structure.

This motivates introducing an effective noise coordinate,
$\gamma_{\mathrm{eff}}$, to organize structure not reflected in a single RB parameter $p$.



## Simple spectral view

A fast Fourier transform (FFT) is a compact way to ask whether the residuals
contain a preferred oscillatory structure rather than only broadband noise.


In [ ]:

residuals_centered = residuals - np.mean(residuals)
fft_vals = np.fft.rfft(residuals_centered)
fft_freqs = np.fft.rfftfreq(len(residuals_centered), d=1.0)

# Ignore zero-frequency component for a clearer view
fft_mag = np.abs(fft_vals)

plt.figure(figsize=(8, 4.6))
plt.plot(fft_freqs[1:], fft_mag[1:], marker="o")
plt.xlabel("Frequency index")
plt.ylabel("FFT magnitude")
plt.title("Spectral view of RB residual structure")
plt.tight_layout()
plt.show()

dominant_idx = np.argmax(fft_mag[1:]) + 1
dominant_freq = fft_freqs[dominant_idx]
dominant_mag = fft_mag[dominant_idx]

dominant_freq, dominant_mag



## Compare residual structure to the injected coherent component

Since the synthetic decay includes a weak coherent oscillatory term, we can compare
the residual shape to that component directly. This is not a full identification procedure,
but it illustrates the idea that residual structure can carry physically interpretable content.


In [ ]:

coherent_component = coherent_amp * np.cos(coherent_freq * m) * np.exp(-0.015 * m)

# Normalize for shape comparison only
coherent_component_norm = coherent_component / np.max(np.abs(coherent_component))
residuals_smooth_norm = residuals_smooth / np.max(np.abs(residuals_smooth))

plt.figure(figsize=(8, 4.6))
plt.plot(m, residuals_smooth_norm, linewidth=2.2, label="Smoothed residuals (normalized)")
plt.plot(m, coherent_component_norm, linewidth=2.2, label="Injected coherent term (normalized)")
plt.xlabel("Gate depth $m$")
plt.ylabel("Normalized amplitude")
plt.title("Residual shape vs coherent structure")
plt.legend()
plt.tight_layout()
plt.show()



## Sweep the effective noise coordinate

We now vary $\gamma_{\phi}$, which changes $\gamma_{\mathrm{eff}}$, and track:
- the fitted RB decay parameter $p$,
- the residual RMSE,
- the peak residual amplitude.

This shows how a physically motivated coordinate can organize changes in
QCVV-style observables and fit quality.


In [ ]:

gamma_phi_values = np.linspace(0.002, 0.028, 10)

gamma_eff_list = []
p_fit_list = []
rmse_list = []
peak_residual_list = []

for gphi in gamma_phi_values:
    geff = gamma + lam * gphi
    coh_amp = coherent_amp * (1.0 + 8.0 * gphi)

    f = (
        A_true * np.exp(-geff * m) + B_true
        + coh_amp * np.cos(coherent_freq * m) * np.exp(-0.015 * m)
        - drift_amp * (m**2)
    )
    f = f + np.random.normal(0.0, noise_sigma, size=len(m))
    f = np.clip(f, 0.0, 1.0)

    pars, _ = curve_fit(rb_model, m, f, p0=p0, bounds=bounds, maxfev=20000)
    A_tmp, p_tmp, B_tmp = pars
    f_rb = rb_model(m, A_tmp, p_tmp, B_tmp)

    res = f - f_rb
    rmse_tmp = np.sqrt(np.mean(res**2))
    peak_tmp = np.max(np.abs(res))

    gamma_eff_list.append(geff)
    p_fit_list.append(p_tmp)
    rmse_list.append(rmse_tmp)
    peak_residual_list.append(peak_tmp)

gamma_eff_arr = np.array(gamma_eff_list)
p_fit_arr = np.array(p_fit_list)
rmse_arr = np.array(rmse_list)
peak_residual_arr = np.array(peak_residual_list)


In [ ]:
# Map sweep results into TPL labels
consistency_scores = np.maximum(0.0, 1.0 - rmse_arr / 0.02)
tpl_labels = [classify_consistency(score, threshold=0.9) for score in consistency_scores]

list(zip(np.round(gamma_eff_arr, 4), np.round(rmse_arr, 5), tpl_labels))


In [ ]:

plt.figure(figsize=(8, 4.8))
plt.plot(gamma_eff_arr, p_fit_arr, marker="o")
plt.xlabel(r"Effective noise $\gamma_{\mathrm{eff}}$")
plt.ylabel(r"RB decay parameter $p$")
plt.title(r"How $p$ shifts with $\gamma_{\mathrm{eff}}$")
plt.tight_layout()
plt.show()


In [ ]:

plt.figure(figsize=(8, 4.8))
plt.plot(gamma_eff_arr, rmse_arr, marker="o", label="RB fit RMSE")
plt.plot(gamma_eff_arr, peak_residual_arr, marker="s", label="Peak residual amplitude")
plt.xlabel(r"Effective noise $\gamma_{\mathrm{eff}}$")
plt.ylabel("Residual metric")
plt.title("Residual metrics across the noise sweep")
plt.legend()
plt.tight_layout()
plt.show()


## Triplet-Phase-Lock summary

Using the triplet-phase-lock framework:

- **VC** marks regimes where residual structure remains comparatively constrained
- **IA** marks regimes where residual behavior indicates stronger mismatch beyond the simple RB model

In this notebook, the framework acts as a compact classification layer on top of the physical residual analysis.



## Takeaway

Residual structure in RB fits is useful signal, not just nuisance.

In this synthetic Rydberg example:
- RB captures the leading decay trend,
- but residuals retain coherent, structured behavior,
- and an effective coordinate such as
  \[
  \gamma_{\mathrm{eff}} = \gamma + \lambda \gamma_{\phi}
  \]
  helps organize how fit quality and residual amplitude change.

This suggests a broader QCVV lesson:

> a single RB decay parameter may summarize the gate, but structured residuals can still
> reveal physically meaningful noise mechanisms.
